# KUKA Feature Selection Experiments

This notebook is organized into clearly separated parts:

- **Part A — Shared setup and reusable helpers**
- **Part B — AE feature-selection experiments**
- **Part C — AAE baseline feature-selection experiments**


In [16]:
from pathlib import Path
import sys
import random

import numpy as np
import pandas as pd

import torch
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "dataset").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from dataset.kuka_splits import split_kuka_numpy
from dataset.kuka_dataset import KukaDataset
from processing.kuka_processing import KukaPreprocessor
from models.AE import AE
from metrics import compute_pr_auc

## Part A.1 — Configuration

In [18]:
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

SEED = 123
BATCH_SIZE = 1024
LATENT_DIM = 4
LR = 1e-3
EPOCHS = 50          # increase for final runs, e.g. 300 or 1500
TOP_K = 5
F_BETA = 2.0

CONSTANT_EPS = 1e-8
CORR_THRESHOLD = 0.98
M_VALUES = [10, 20, 40, 60]

DEVICE = torch.device("cpu")
DEVICE

device(type='cpu')

In [19]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

## Part A.2 — Load dataset split

In [20]:
splits = split_kuka_numpy(DATA_DIR, seed=SEED, binary_labels=True)

print("x_train:", splits.x_train.shape, "y_train:", splits.y_train.shape, np.unique(splits.y_train, return_counts=True))
print("x_val:  ", splits.x_val.shape,   "y_val:  ", splits.y_val.shape,   np.unique(splits.y_val, return_counts=True))
print("x_test: ", splits.x_test.shape,  "y_test: ", splits.y_test.shape,  np.unique(splits.y_test, return_counts=True))

n_features_raw = splits.x_train.shape[1]
print("Raw feature count:", n_features_raw)

x_train: (163654, 86) y_train: (163654,) (array([0], dtype=int64), array([163654], dtype=int64))
x_val:   (55837, 86) y_val:   (55837,) (array([0, 1], dtype=int64), array([35068, 20769], dtype=int64))
x_test:  (55839, 86) y_test:  (55839,) (array([0, 1], dtype=int64), array([35070, 20769], dtype=int64))
Raw feature count: 86


## Part A.3 — Load feature names

In [21]:
def load_feature_names(data_dir: Path, n_features: int) -> np.ndarray:
    path = data_dir / "KukaColumnNames.npy"

    if not path.exists():
        print("KukaColumnNames.npy not found. Using generic feature names.")
        return np.array([f"feature_{i}" for i in range(n_features)], dtype=object)

    names = np.load(path, allow_pickle=True).astype(str)

    if len(names) == n_features:
        return names

    if len(names) == n_features + 1:
        print("Feature-name file has one extra column. Assuming last entry is label and dropping it.")
        return names[:-1]

    print(f"Unexpected number of feature names: {len(names)} for {n_features} features. Using generic feature names.")
    return np.array([f"feature_{i}" for i in range(n_features)], dtype=object)

feature_names = load_feature_names(DATA_DIR, n_features_raw)
assert len(feature_names) == n_features_raw

pd.DataFrame({"feature_index": np.arange(n_features_raw), "feature_name": feature_names}).head(10)

Feature-name file has one extra column. Assuming last entry is label and dropping it.


,feature_index,feature_name
0,0,action
1,1,machine_nameKuka Robot_apparent_power
2,2,machine_nameKuka Robot_current
3,3,machine_nameKuka Robot_frequency
4,4,machine_nameKuka Robot_phase_angle
5,5,machine_nameKuka Robot_power
6,6,machine_nameKuka Robot_power_factor
7,7,machine_nameKuka Robot_reactive_power
8,8,machine_nameKuka Robot_voltage
9,9,sensor_id1_AccX


## Part A.4 — Shared AE utility functions

These functions support the AE experiments in Part B.


In [22]:
def make_dataloaders_for_features(selected_features, batch_size=BATCH_SIZE, normalization=True):
    """Build DataLoaders for a selected subset of original feature indices."""
    selected_features = np.asarray(selected_features, dtype=int)

    x_train_raw = splits.x_train[:, selected_features]
    x_val_raw = splits.x_val[:, selected_features]
    x_test_raw = splits.x_test[:, selected_features]

    preprocessor = KukaPreprocessor(
        normalization=normalization,
        remove_constant=False,
        corr_threshold=None,
        clip_percentiles=None,
    )

    x_train = preprocessor.fit_transform(x_train_raw)
    x_val = preprocessor.transform(x_val_raw)
    x_test = preprocessor.transform(x_test_raw)

    train_dataset = KukaDataset(x_train, splits.y_train)
    val_dataset = KukaDataset(x_val, splits.y_val)
    test_dataset = KukaDataset(x_test, splits.y_test)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader, preprocessor


def ae_error_fn(pred, x, top_k=TOP_K):
    k = min(int(top_k), x.shape[1])
    return (pred - x).pow(2).topk(k, dim=1).values.mean(dim=1)


def train_ae_model(model, train_loader, device=DEVICE, lr=LR, epochs=EPOCHS):
    loss_fn = torch.nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    model.to(device)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for x, _ in train_loader:
            x = x.to(device)
            optimizer.zero_grad()
            recon = model(x)
            loss = loss_fn(recon, x)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        epoch_loss = running_loss / len(train_loader)
        history.append(epoch_loss)

        if epoch == 0 or (epoch + 1) % max(1, epochs // 5) == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch + 1:4d}/{epochs}: loss={epoch_loss:.6f}")

    return history

@torch.no_grad()
def compute_reconstruction_error_scores(model, dataloader, recon_top_k):
    recon_scores = []
    labels = []

    model.eval()
    for x, y in dataloader:
        recon = model(x)
        
        if isinstance(recon, tuple):
            recon = recon[0]

        # reconstruction error score
        recon_score = (recon - x).pow(2).topk(recon_top_k, dim=1).values.mean(dim=1)

        recon_scores.append(recon_score.cpu())
        labels.append(y.cpu())

    recon_scores = torch.cat(recon_scores)
    labels = torch.cat(labels)

    return recon_scores, labels


def run_ae_experiment(experiment_name, selected_features, epochs=EPOCHS, seed=SEED):
    """Train and evaluate an AE on a specific feature subset."""
    selected_features = np.asarray(selected_features, dtype=int)

    if len(selected_features) == 0:
        raise ValueError(f"{experiment_name}: selected feature subset is empty")

    print("=" * 80)
    print(f"Experiment: {experiment_name}")
    print(f"Selected features: {len(selected_features)}")

    set_seed(seed)

    train_loader, val_loader, test_loader, preprocessor = make_dataloaders_for_features(selected_features)

    input_dim = len(selected_features)
    model = AE(input_dim, LATENT_DIM).to(DEVICE)

    history = train_ae_model(model, train_loader, epochs=epochs)

    
    scores, labels = compute_reconstruction_error_scores(model, val_loader, TOP_K)

    pr_auc = compute_pr_auc(
        scores=scores,
        labels=labels
    )

    row = {
        "experiment": experiment_name,
        "n_features": int(len(selected_features)),
        "pr_auc": pr_auc
    }

    print(pd.Series(row))

    return {
        "row": row,
        "model": model,
        "history": history,
        "selected_features": selected_features,
        "preprocessor": preprocessor,
        "labels": labels,
    }

## Part A.5 — Shared feature-selection functions

In [23]:
def select_all_features(n_features: int) -> np.ndarray:
    return np.arange(n_features, dtype=int)


def select_nonconstant_features(x_train, eps=CONSTANT_EPS) -> np.ndarray:
    std = np.std(x_train, axis=0)
    return np.flatnonzero(std > eps)


def correlation_prune_features(x_train, candidate_features, threshold=CORR_THRESHOLD) -> np.ndarray:
    """Greedy correlation pruning in the order given by candidate_features."""
    candidate_features = np.asarray(candidate_features, dtype=int)
    x = x_train[:, candidate_features]

    if x.shape[1] <= 1:
        return candidate_features

    corr = np.corrcoef(x, rowvar=False)
    corr = np.nan_to_num(np.abs(corr), nan=0.0, posinf=0.0, neginf=0.0)

    keep_local = []
    for j in range(x.shape[1]):
        if not keep_local:
            keep_local.append(j)
            continue

        max_corr = np.max(corr[j, keep_local])
        if max_corr <= threshold:
            keep_local.append(j)

    return candidate_features[np.array(keep_local, dtype=int)]


def ks_statistic_1d(a, b):
    """Two-sample KS statistic D. Does not require scipy."""
    a = np.sort(np.asarray(a, dtype=float))
    b = np.sort(np.asarray(b, dtype=float))

    if len(a) == 0 or len(b) == 0:
        return np.nan

    values = np.concatenate([a, b])
    cdf_a = np.searchsorted(a, values, side="right") / len(a)
    cdf_b = np.searchsorted(b, values, side="right") / len(b)
    return float(np.max(np.abs(cdf_a - cdf_b)))


def rank_features_by_ks(x_val, y_val, candidate_features):
    candidate_features = np.asarray(candidate_features, dtype=int)
    y_val = np.asarray(y_val).astype(int)

    normal_mask = y_val == 0
    anomaly_mask = y_val == 1

    if normal_mask.sum() == 0 or anomaly_mask.sum() == 0:
        raise ValueError("Validation split must contain both normal and anomaly samples for KS ranking.")

    scores = []
    for f in candidate_features:
        scores.append(ks_statistic_1d(x_val[normal_mask, f], x_val[anomaly_mask, f]))

    scores = np.asarray(scores)
    order = np.argsort(scores)[::-1]

    return candidate_features[order], scores[order]


def rank_features_by_ae_reconstruction_separation(model, val_loader, candidate_features, eps=1e-8):
    """
    Rank selected input features by validation reconstruction-error separation:

        r_j = (mean_anomaly_error_j - mean_normal_error_j) / (std_normal_error_j + eps)

    candidate_features maps local model columns back to original feature indices.
    """
    model.eval()

    normal_errors = []
    anomaly_errors = []

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE)
            recon = model(x)
            feature_err = (recon - x).pow(2).cpu()

            y = y.cpu()
            if (y == 0).any():
                normal_errors.append(feature_err[y == 0])
            if (y == 1).any():
                anomaly_errors.append(feature_err[y == 1])

    if not normal_errors or not anomaly_errors:
        raise ValueError("Validation split must contain both normal and anomaly samples.")

    normal_errors = torch.cat(normal_errors, dim=0)
    anomaly_errors = torch.cat(anomaly_errors, dim=0)

    normal_mean = normal_errors.mean(dim=0)
    normal_std = normal_errors.std(dim=0)
    anomaly_mean = anomaly_errors.mean(dim=0)

    separation = ((anomaly_mean - normal_mean) / (normal_std + eps)).numpy()
    order = np.argsort(separation)[::-1]

    candidate_features = np.asarray(candidate_features, dtype=int)
    return candidate_features[order], separation[order]

# Part B — AE feature-selection experiments

## Part B.1 — Build AE feature subsets


In [19]:
all_features = select_all_features(n_features_raw)
nonconstant_features = select_nonconstant_features(splits.x_train, eps=CONSTANT_EPS)
nonconstant_corr_features = correlation_prune_features(
    splits.x_train,
    nonconstant_features,
    threshold=CORR_THRESHOLD,
)

print("All features:                 ", len(all_features))
print("After near-constant removal:  ", len(nonconstant_features))
print("After correlation pruning:    ", len(nonconstant_corr_features))

removed_constant = np.setdiff1d(all_features, nonconstant_features)
removed_corr = np.setdiff1d(nonconstant_features, nonconstant_corr_features)

print("Near-constant removed:")
display(pd.DataFrame({"feature_index": removed_constant, "feature_name": feature_names[removed_constant]}))

print("Correlation-pruned removed, first 20:")
display(pd.DataFrame({"feature_index": removed_corr[:20], "feature_name": feature_names[removed_corr[:20]]}))

All features:                  86
After near-constant removal:   82
After correlation pruning:     79
Near-constant removed:


,feature_index,feature_name
0,30,sensor_id2_temp
1,63,sensor_id5_temp
2,74,sensor_id6_temp
3,85,sensor_id7_temp


Correlation-pruned removed, first 20:


,feature_index,feature_name
0,5,machine_nameKuka Robot_power
1,6,machine_nameKuka Robot_power_factor
2,80,sensor_id7_GyroZ


In [20]:
# KS ranking uses validation labels only.
# Use nonconstant features as candidates to avoid selecting degenerate channels.

ks_ranked_features, ks_scores = rank_features_by_ks(
    splits.x_val,
    splits.y_val,
    candidate_features=nonconstant_features,
)

ks_table = pd.DataFrame({
    "rank": np.arange(1, len(ks_ranked_features) + 1),
    "feature_index": ks_ranked_features,
    "feature_name": feature_names[ks_ranked_features],
    "ks_score": ks_scores,
})

ks_table.head(20)

,rank,feature_index,feature_name,ks_score
0,1,37,sensor_id3_q1,0.444422
1,2,17,sensor_id1_q3,0.413103
2,3,15,sensor_id1_q1,0.411799
3,4,72,sensor_id6_q3,0.395636
4,5,71,sensor_id6_q2,0.359868
5,6,38,sensor_id3_q2,0.357046
6,7,8,machine_nameKuka Robot_voltage,0.335216
7,8,50,sensor_id4_q3,0.333629
8,9,4,machine_nameKuka Robot_phase_angle,0.308354
9,10,6,machine_nameKuka Robot_power_factor,0.308327


## Part B.2 — Run AE experiments for all, nonconstant, correlation-pruned and KS-selected features

In [21]:
experiment_outputs = []
selected_feature_tables = {}

out_all = run_ae_experiment("all_features", all_features, epochs=EPOCHS, seed=SEED)
experiment_outputs.append(out_all)
selected_feature_tables["all_features"] = all_features

out_nc = run_ae_experiment("nonconstant", nonconstant_features, epochs=EPOCHS, seed=SEED)
experiment_outputs.append(out_nc)
selected_feature_tables["nonconstant"] = nonconstant_features

out_nc_corr = run_ae_experiment("nonconstant_corr", nonconstant_corr_features, epochs=EPOCHS, seed=SEED)
experiment_outputs.append(out_nc_corr)
selected_feature_tables["nonconstant_corr"] = nonconstant_corr_features

for m in M_VALUES:
    m_eff = min(m, len(ks_ranked_features))
    selected = ks_ranked_features[:m_eff]
    name = f"top_{m_eff}_ks"
    out = run_ae_experiment(name, selected, epochs=EPOCHS, seed=SEED)
    experiment_outputs.append(out)
    selected_feature_tables[name] = selected

Experiment: all_features
Selected features: 86
Epoch    1/50: loss=0.062107
Epoch   10/50: loss=0.018654
Epoch   20/50: loss=0.016381
Epoch   30/50: loss=0.014690
Epoch   40/50: loss=0.013837
Epoch   50/50: loss=0.013143
experiment    all_features
n_features              86
pr_auc            0.722222
dtype: object
Experiment: nonconstant
Selected features: 82
Epoch    1/50: loss=0.061263
Epoch   10/50: loss=0.017505
Epoch   20/50: loss=0.015494
Epoch   30/50: loss=0.012855
Epoch   40/50: loss=0.011900
Epoch   50/50: loss=0.011292
experiment    nonconstant
n_features             82
pr_auc            0.67768
dtype: object
Experiment: nonconstant_corr
Selected features: 79
Epoch    1/50: loss=0.065274
Epoch   10/50: loss=0.019475
Epoch   20/50: loss=0.016421
Epoch   30/50: loss=0.014584
Epoch   40/50: loss=0.013561
Epoch   50/50: loss=0.012818
experiment    nonconstant_corr
n_features                  79
pr_auc                0.711392
dtype: object
Experiment: top_10_ks
Selected features:

## Part B.3 — Rank features by AE reconstruction-error separation

This method first trains an AE on the correlation-pruned candidate feature space, then ranks features using validation reconstruction-error separation:

$$
r_j = \frac{\mu_{anom,j}-\mu_{normal,j}}{\sigma_{normal,j}+\epsilon}.
$$


In [22]:
base_model_for_recon_ranking = out_nc_corr["model"]
base_candidate_features = nonconstant_corr_features

_, val_loader_for_base, _, _ = make_dataloaders_for_features(base_candidate_features)

ae_ranked_features, ae_sep_scores = rank_features_by_ae_reconstruction_separation(
    base_model_for_recon_ranking,
    val_loader_for_base,
    candidate_features=base_candidate_features,
)

ae_sep_table = pd.DataFrame({
    "rank": np.arange(1, len(ae_ranked_features) + 1),
    "feature_index": ae_ranked_features,
    "feature_name": feature_names[ae_ranked_features],
    "ae_reconstruction_separation": ae_sep_scores,
})

ae_sep_table.head(20)

,rank,feature_index,feature_name,ae_reconstruction_separation
0,1,50,sensor_id4_q3,12.993514
1,2,48,sensor_id4_q1,10.264585
2,3,49,sensor_id4_q2,6.606115
3,4,51,sensor_id4_q4,4.698593
4,5,60,sensor_id5_q2,2.014117
5,6,62,sensor_id5_q4,1.920820
6,7,38,sensor_id3_q2,1.320322
7,8,83,sensor_id7_q3,0.863053
8,9,82,sensor_id7_q2,0.700471
9,10,84,sensor_id7_q4,0.660610


## Part B.4 — Run AE experiments for reconstruction-separation selected features

This cell evaluates top-$m$ subsets selected by the AE reconstruction-error separation ranking.


In [23]:
for m in M_VALUES:
    m_eff = min(m, len(ae_ranked_features))
    selected = ae_ranked_features[:m_eff]
    name = f"top_{m_eff}_ae_recon_sep"
    out = run_ae_experiment(name, selected, epochs=EPOCHS, seed=SEED)
    experiment_outputs.append(out)
    selected_feature_tables[name] = selected

Experiment: top_10_ae_recon_sep
Selected features: 10
Epoch    1/50: loss=0.110249
Epoch   10/50: loss=0.011237
Epoch   20/50: loss=0.008962
Epoch   30/50: loss=0.008278
Epoch   40/50: loss=0.007318
Epoch   50/50: loss=0.007002
experiment    top_10_ae_recon_sep
n_features                     10
pr_auc                   0.724544
dtype: object
Experiment: top_20_ae_recon_sep
Selected features: 20
Epoch    1/50: loss=0.080117
Epoch   10/50: loss=0.019911
Epoch   20/50: loss=0.015579
Epoch   30/50: loss=0.013996
Epoch   40/50: loss=0.013073
Epoch   50/50: loss=0.011484
experiment    top_20_ae_recon_sep
n_features                     20
pr_auc                   0.776479
dtype: object
Experiment: top_40_ae_recon_sep
Selected features: 40
Epoch    1/50: loss=0.076402
Epoch   10/50: loss=0.021831
Epoch   20/50: loss=0.018337
Epoch   30/50: loss=0.016039
Epoch   40/50: loss=0.014747
Epoch   50/50: loss=0.013819
experiment    top_40_ae_recon_sep
n_features                     40
pr_auc          

## Part B.5 — AE results summary


In [26]:
results_df = pd.DataFrame([out["row"] for out in experiment_outputs])

results_df.sort_values("pr_auc", ascending=False)

,experiment,n_features,pr_auc
9,top_40_ae_recon_sep,40,0.892075
4,top_20_ks,20,0.832197
5,top_40_ks,40,0.808333
3,top_10_ks,10,0.797754
8,top_20_ae_recon_sep,20,0.776479
6,top_60_ks,60,0.762130
10,top_60_ae_recon_sep,60,0.731540
7,top_10_ae_recon_sep,10,0.724544
0,all_features,86,0.722222
2,nonconstant_corr,79,0.711392


## Part B.6 — Save AE feature-selection outputs


In [27]:
results_path = RESULTS_DIR / "feature_selection_results.csv"
ks_path = RESULTS_DIR / "feature_selection_ks_ranking.csv"
ae_sep_path = RESULTS_DIR / "feature_selection_ae_recon_sep_ranking.csv"
selected_path = RESULTS_DIR / "feature_selection_selected_features.csv"

results_df.to_csv(results_path, index=False)
ks_table.to_csv(ks_path, index=False)
ae_sep_table.to_csv(ae_sep_path, index=False)

selected_rows = []
for experiment, indices in selected_feature_tables.items():
    for rank, idx in enumerate(indices, start=1):
        selected_rows.append({
            "experiment": experiment,
            "rank": rank,
            "feature_index": int(idx),
            "feature_name": str(feature_names[idx]),
        })

selected_df = pd.DataFrame(selected_rows)
selected_df.to_csv(selected_path, index=False)

print("Saved results.")

Saved results.


## Part B.7 — Inspect AE selected features

In [29]:
EXPERIMENT_TO_INSPECT = "top_40_ae_recon_sep"

if EXPERIMENT_TO_INSPECT in selected_feature_tables:
    idx = selected_feature_tables[EXPERIMENT_TO_INSPECT]
    display(pd.DataFrame({
        "rank": np.arange(1, len(idx) + 1),
        "feature_index": idx,
        "feature_name": feature_names[idx],
    }))
else:
    print("Available experiments:")
    print(list(selected_feature_tables.keys()))

,rank,feature_index,feature_name
0,1,50,sensor_id4_q3
1,2,48,sensor_id4_q1
2,3,49,sensor_id4_q2
3,4,51,sensor_id4_q4
4,5,60,sensor_id5_q2
5,6,62,sensor_id5_q4
6,7,38,sensor_id3_q2
7,8,83,sensor_id7_q3
8,9,82,sensor_id7_q2
9,10,84,sensor_id7_q4


# Part C — AAE baseline feature-selection experiments

This part implements feature selection for the Adversarial Autoencoder.


## Part C.1 — AAE imports and configuration

In [25]:
from models.AAE import AAE
from train_aae import train as train_aae_model

AAE_SEED = SEED
AAE_BATCH_SIZE = BATCH_SIZE
AAE_LATENT_DIM = LATENT_DIM
AAE_EPOCHS = EPOCHS
AAE_TOP_K = TOP_K

AAE_LR_AE = 1e-3
AAE_LR_D = 5e-5
AAE_LR_G = 1e-4

AAE_RESULTS_PATH = RESULTS_DIR / "feature_selection_aae_baseline_results.csv"
AAE_SELECTED_PATH = RESULTS_DIR / "feature_selection_aae_selected_features.csv"
AAE_KS_PATH = RESULTS_DIR / "feature_selection_aae_ks_ranking.csv"
AAE_RECON_SEP_PATH = RESULTS_DIR / "feature_selection_aae_recon_sep_ranking.csv"


## Part C.2 — Load AAE-specific split and feature names

In [26]:
aae_splits = split_kuka_numpy(DATA_DIR, seed=AAE_SEED, binary_labels=True)

aae_n_features_raw = aae_splits.x_train.shape[1]
aae_feature_names = load_feature_names(DATA_DIR, aae_n_features_raw)

print("AAE x_train:", aae_splits.x_train.shape, "y_train:", aae_splits.y_train.shape, np.unique(aae_splits.y_train, return_counts=True))
print("AAE x_val:  ", aae_splits.x_val.shape, "y_val:  ", aae_splits.y_val.shape, np.unique(aae_splits.y_val, return_counts=True))
print("AAE x_test: ", aae_splits.x_test.shape, "y_test: ", aae_splits.y_test.shape, np.unique(aae_splits.y_test, return_counts=True))


Feature-name file has one extra column. Assuming last entry is label and dropping it.
AAE x_train: (163654, 86) y_train: (163654,) (array([0], dtype=int64), array([163654], dtype=int64))
AAE x_val:   (55837, 86) y_val:   (55837,) (array([0, 1], dtype=int64), array([35068, 20769], dtype=int64))
AAE x_test:  (55839, 86) y_test:  (55839,) (array([0, 1], dtype=int64), array([35070, 20769], dtype=int64))


## Part C.3 — AAE-specific dataloaders and scoring helpers

The AAE baseline score is reconstruction-only:

$$
s_{AAE,rec}(x)=\frac{1}{k}\sum_{r=1}^{k} e_{(r)}(x).
$$


In [27]:
def make_aae_dataloaders_for_features(selected_features, batch_size=AAE_BATCH_SIZE, normalization=True):
    # Build AAE DataLoaders for a selected subset of original feature indices.
    selected_features = np.asarray(selected_features, dtype=int)

    x_train_raw = aae_splits.x_train[:, selected_features]
    x_val_raw = aae_splits.x_val[:, selected_features]
    x_test_raw = aae_splits.x_test[:, selected_features]

    preprocessor = KukaPreprocessor(
        normalization=normalization,
        remove_constant=False,
        corr_threshold=None,
        clip_percentiles=None,
    )

    x_train = preprocessor.fit_transform(x_train_raw)
    x_val = preprocessor.transform(x_val_raw)
    x_test = preprocessor.transform(x_test_raw)

    train_dataset = KukaDataset(x_train, aae_splits.y_train)
    val_dataset = KukaDataset(x_val, aae_splits.y_val)
    test_dataset = KukaDataset(x_test, aae_splits.y_test)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader, preprocessor


@torch.no_grad()
def compute_aae_reconstruction_scores(model, dataloader, recon_top_k=AAE_TOP_K):
    # Compute top-k reconstruction scores for an AAE model without fusion.
    scores = []
    labels = []

    model.eval()
    device = next(model.parameters()).device

    for x, y in dataloader:
        x = x.float().to(device)
        recon, _ = model(x)
        k = min(int(recon_top_k), x.shape[1])
        score = (recon - x).pow(2).topk(k, dim=1).values.mean(dim=1)

        scores.append(score.detach().cpu())
        labels.append(y.cpu())

    return torch.cat(scores), torch.cat(labels)


def run_aae_baseline_experiment(experiment_name, selected_features, epochs=AAE_EPOCHS, seed=AAE_SEED):
    # Train and evaluate an AAE baseline on a specific feature subset using reconstruction score only.
    selected_features = np.asarray(selected_features, dtype=int)

    if len(selected_features) == 0:
        raise ValueError(f"{experiment_name}: selected feature subset is empty")

    print("=" * 80)
    print(f"AAE baseline experiment: {experiment_name}")
    print(f"Selected features: {len(selected_features)}")

    set_seed(seed)

    train_loader, val_loader, test_loader, preprocessor = make_aae_dataloaders_for_features(selected_features)

    input_dim = len(selected_features)
    model = AAE(input_dim, AAE_LATENT_DIM).to(DEVICE)

    history = train_aae_model(
        model,
        train_loader,
        epochs=epochs,
        device=DEVICE,
        lr_ae=AAE_LR_AE,
        lr_g=AAE_LR_G,
        lr_d=AAE_LR_D,
    )

    scores, labels = compute_aae_reconstruction_scores(model, val_loader, AAE_TOP_K)
    pr_auc = compute_pr_auc(scores=scores, labels=labels)

    row = {
        "experiment": experiment_name,
        "model": "AAE_reconstruction_only",
        "n_features": int(len(selected_features)),
        "pr_auc": pr_auc,
    }

    print(pd.Series(row))

    return {
        "row": row,
        "model": model,
        "history": history,
        "selected_features": selected_features,
        "preprocessor": preprocessor,
        "scores": scores,
        "labels": labels,
    }


## Part C.4 — Build AAE feature subsets: all, nonconstant, correlation-pruned and KS-ranked

In [28]:
aae_all_features = select_all_features(aae_n_features_raw)
aae_nonconstant_features = select_nonconstant_features(aae_splits.x_train, eps=CONSTANT_EPS)
aae_nonconstant_corr_features = correlation_prune_features(
    aae_splits.x_train,
    aae_nonconstant_features,
    threshold=CORR_THRESHOLD,
)

print("AAE all features:                 ", len(aae_all_features))
print("AAE after near-constant removal:  ", len(aae_nonconstant_features))
print("AAE after correlation pruning:    ", len(aae_nonconstant_corr_features))

aae_ks_ranked_features, aae_ks_scores = rank_features_by_ks(
    aae_splits.x_val,
    aae_splits.y_val,
    candidate_features=aae_nonconstant_features,
)

aae_ks_table = pd.DataFrame({
    "rank": np.arange(1, len(aae_ks_ranked_features) + 1),
    "feature_index": aae_ks_ranked_features,
    "feature_name": aae_feature_names[aae_ks_ranked_features],
    "ks_score": aae_ks_scores,
})

aae_selected_feature_tables = {
    "all_features": aae_all_features,
    "nonconstant": aae_nonconstant_features,
    "nonconstant_corr": aae_nonconstant_corr_features,
}

for m in M_VALUES:
    m_eff = min(m, len(aae_ks_ranked_features))
    aae_selected_feature_tables[f"top_{m_eff}_ks"] = aae_ks_ranked_features[:m_eff]

aae_ks_table.head(20)


AAE all features:                  86
AAE after near-constant removal:   82
AAE after correlation pruning:     79


,rank,feature_index,feature_name,ks_score
0,1,37,sensor_id3_q1,0.444422
1,2,17,sensor_id1_q3,0.413103
2,3,15,sensor_id1_q1,0.411799
3,4,72,sensor_id6_q3,0.395636
4,5,71,sensor_id6_q2,0.359868
5,6,38,sensor_id3_q2,0.357046
6,7,8,machine_nameKuka Robot_voltage,0.335216
7,8,50,sensor_id4_q3,0.333629
8,9,4,machine_nameKuka Robot_phase_angle,0.308354
9,10,6,machine_nameKuka Robot_power_factor,0.308327


## Part C.5 — Run AAE baseline experiments for all, nonconstant, correlation-pruned and KS-selected features

In [29]:
aae_experiment_outputs = []

aae_initial_experiments_to_run = [
    "all_features",
    "nonconstant",
    "nonconstant_corr",
    *[f"top_{min(m, len(aae_ks_ranked_features))}_ks" for m in M_VALUES],
]

# Remove duplicate names in case M_VALUES contains values larger than the ranking length.
aae_initial_experiments_to_run = list(dict.fromkeys(aae_initial_experiments_to_run))

for experiment_name in aae_initial_experiments_to_run:
    selected = aae_selected_feature_tables[experiment_name]
    out = run_aae_baseline_experiment(experiment_name, selected, epochs=AAE_EPOCHS, seed=AAE_SEED)
    aae_experiment_outputs.append(out)


AAE baseline experiment: all_features
Selected features: 86
Epoch [1/50] Recon Loss: 0.062115 | D Loss: 2.061319 | G Loss: 1.168327 
Epoch [2/50] Recon Loss: 0.027127 | D Loss: 2.051675 | G Loss: 1.154286 
Epoch [3/50] Recon Loss: 0.025064 | D Loss: 1.851864 | G Loss: 1.335892 
Epoch [4/50] Recon Loss: 0.022908 | D Loss: 1.857308 | G Loss: 1.317682 
Epoch [5/50] Recon Loss: 0.022040 | D Loss: 1.917396 | G Loss: 1.242625 
Epoch [6/50] Recon Loss: 0.021069 | D Loss: 1.957298 | G Loss: 1.183175 
Epoch [7/50] Recon Loss: 0.020738 | D Loss: 1.962135 | G Loss: 1.162479 
Epoch [8/50] Recon Loss: 0.019937 | D Loss: 1.950720 | G Loss: 1.180017 
Epoch [9/50] Recon Loss: 0.020070 | D Loss: 1.926778 | G Loss: 1.214472 
Epoch [10/50] Recon Loss: 0.019437 | D Loss: 1.918271 | G Loss: 1.220688 
Epoch [11/50] Recon Loss: 0.018253 | D Loss: 1.924857 | G Loss: 1.206010 
Epoch [12/50] Recon Loss: 0.017674 | D Loss: 1.937325 | G Loss: 1.192585 
Epoch [13/50] Recon Loss: 0.017334 | D Loss: 1.947445 | G Los

## Part C.6 — Rank features by AAE reconstruction-error separation

This ranking is based on the AAE model trained on the correlation-pruned feature set.

$$
r_j = \frac{\mu_{anom,j}-\mu_{normal,j}}{\sigma_{normal,j}+\epsilon}.
$$


In [30]:
@torch.no_grad()
def rank_features_by_aae_reconstruction_separation(model, val_loader, candidate_features, eps=1e-8):
    # Rank local AAE input columns by reconstruction-error separation and map them to original feature indices.
    model.eval()
    device = next(model.parameters()).device

    normal_errors = []
    anomaly_errors = []

    for x, y in val_loader:
        x = x.float().to(device)
        recon, _ = model(x)
        feature_err = (recon - x).pow(2).detach().cpu()

        y = y.cpu()
        if (y == 0).any():
            normal_errors.append(feature_err[y == 0])
        if (y == 1).any():
            anomaly_errors.append(feature_err[y == 1])

    if not normal_errors or not anomaly_errors:
        raise ValueError("Validation split must contain both normal and anomaly samples.")

    normal_errors = torch.cat(normal_errors, dim=0)
    anomaly_errors = torch.cat(anomaly_errors, dim=0)

    normal_mean = normal_errors.mean(dim=0)
    normal_std = normal_errors.std(dim=0)
    anomaly_mean = anomaly_errors.mean(dim=0)

    separation = ((anomaly_mean - normal_mean) / (normal_std + eps)).numpy()
    order = np.argsort(separation)[::-1]

    candidate_features = np.asarray(candidate_features, dtype=int)
    return candidate_features[order], separation[order]


aae_nc_corr_output = next(out for out in aae_experiment_outputs if out["row"]["experiment"] == "nonconstant_corr")
aae_base_model_for_recon_ranking = aae_nc_corr_output["model"]
aae_base_candidate_features = aae_nonconstant_corr_features

_, aae_val_loader_for_base, _, _ = make_aae_dataloaders_for_features(aae_base_candidate_features)

aae_ranked_features, aae_sep_scores = rank_features_by_aae_reconstruction_separation(
    aae_base_model_for_recon_ranking,
    aae_val_loader_for_base,
    candidate_features=aae_base_candidate_features,
)

aae_sep_table = pd.DataFrame({
    "rank": np.arange(1, len(aae_ranked_features) + 1),
    "feature_index": aae_ranked_features,
    "feature_name": aae_feature_names[aae_ranked_features],
    "aae_reconstruction_separation": aae_sep_scores,
})

aae_sep_table.head(20)


,rank,feature_index,feature_name,aae_reconstruction_separation
0,1,50,sensor_id4_q3,13.970358
1,2,48,sensor_id4_q1,9.518894
2,3,49,sensor_id4_q2,6.876550
3,4,51,sensor_id4_q4,3.262555
4,5,38,sensor_id3_q2,2.289487
5,6,72,sensor_id6_q3,0.643837
6,7,62,sensor_id5_q4,0.550349
7,8,83,sensor_id7_q3,0.550144
8,9,70,sensor_id6_q1,0.370463
9,10,73,sensor_id6_q4,0.364268


## Part C.7 — Run AAE baseline experiments for AAE reconstruction-separation selected features

In [31]:
for m in M_VALUES:
    m_eff = min(m, len(aae_ranked_features))
    selected = aae_ranked_features[:m_eff]
    name = f"top_{m_eff}_aae_recon_sep"
    aae_selected_feature_tables[name] = selected

    out = run_aae_baseline_experiment(name, selected, epochs=AAE_EPOCHS, seed=AAE_SEED)
    aae_experiment_outputs.append(out)


AAE baseline experiment: top_10_aae_recon_sep
Selected features: 10
Epoch [1/50] Recon Loss: 0.105396 | D Loss: 2.052872 | G Loss: 0.890058 
Epoch [2/50] Recon Loss: 0.037809 | D Loss: 2.399743 | G Loss: 0.629075 
Epoch [3/50] Recon Loss: 0.028738 | D Loss: 1.859593 | G Loss: 1.048289 
Epoch [4/50] Recon Loss: 0.022916 | D Loss: 1.758019 | G Loss: 1.107571 
Epoch [5/50] Recon Loss: 0.019253 | D Loss: 1.910473 | G Loss: 0.948548 
Epoch [6/50] Recon Loss: 0.015614 | D Loss: 1.813195 | G Loss: 1.044460 
Epoch [7/50] Recon Loss: 0.013620 | D Loss: 1.777558 | G Loss: 1.070883 
Epoch [8/50] Recon Loss: 0.011834 | D Loss: 1.786211 | G Loss: 1.052671 
Epoch [9/50] Recon Loss: 0.010390 | D Loss: 1.808551 | G Loss: 1.029460 
Epoch [10/50] Recon Loss: 0.009698 | D Loss: 1.833837 | G Loss: 1.008321 
Epoch [11/50] Recon Loss: 0.009166 | D Loss: 1.863404 | G Loss: 0.986440 
Epoch [12/50] Recon Loss: 0.008694 | D Loss: 1.894221 | G Loss: 0.959121 
Epoch [13/50] Recon Loss: 0.008154 | D Loss: 1.914891

## Part C.8 — AAE baseline results summary

In [32]:
aae_results_df = pd.DataFrame([out["row"] for out in aae_experiment_outputs])
aae_results_df.sort_values("pr_auc", ascending=False)


,experiment,model,n_features,pr_auc
3,top_10_ks,AAE_reconstruction_only,10,0.893346
8,top_20_aae_recon_sep,AAE_reconstruction_only,20,0.872348
7,top_10_aae_recon_sep,AAE_reconstruction_only,10,0.862981
9,top_40_aae_recon_sep,AAE_reconstruction_only,40,0.817864
6,top_60_ks,AAE_reconstruction_only,60,0.796203
5,top_40_ks,AAE_reconstruction_only,40,0.787968
10,top_60_aae_recon_sep,AAE_reconstruction_only,60,0.784920
4,top_20_ks,AAE_reconstruction_only,20,0.782199
0,all_features,AAE_reconstruction_only,86,0.737292
2,nonconstant_corr,AAE_reconstruction_only,79,0.716179


## Part C.9 — Save AAE feature-selection outputs

The AAE output files are separate from the AE output files.


In [34]:
aae_results_df.to_csv(AAE_RESULTS_PATH, index=False)

aae_ks_table.to_csv(AAE_KS_PATH, index=False)
aae_sep_table.to_csv(AAE_RECON_SEP_PATH, index=False)

aae_selected_rows = []
for experiment, indices in aae_selected_feature_tables.items():
    for rank, idx in enumerate(indices, start=1):
        aae_selected_rows.append({
            "experiment": experiment,
            "rank": rank,
            "feature_index": int(idx),
            "feature_name": str(aae_feature_names[idx]),
        })

aae_selected_df = pd.DataFrame(aae_selected_rows)
aae_selected_df.to_csv(AAE_SELECTED_PATH, index=False)

print("Saved AAE baseline feature-selection outputs.")


Saved AAE baseline feature-selection outputs.


## Part C.10 — Inspect AAE selected features

Use this cell to inspect the feature names selected by any AAE experiment.


In [ ]:
AAE_EXPERIMENT_TO_INSPECT = "top_10_ks_sep"

if AAE_EXPERIMENT_TO_INSPECT in aae_selected_feature_tables:
    idx = aae_selected_feature_tables[AAE_EXPERIMENT_TO_INSPECT]
    display(pd.DataFrame({
        "rank": np.arange(1, len(idx) + 1),
        "feature_index": idx,
        "feature_name": aae_feature_names[idx],
    }))
else:
    print(f"Unknown AAE experiment: {AAE_EXPERIMENT_TO_INSPECT}")
    print("Available AAE experiments:")
    print(sorted(aae_selected_feature_tables.keys()))


,rank,feature_index,feature_name
0,1,50,sensor_id4_q3
1,2,48,sensor_id4_q1
2,3,49,sensor_id4_q2
3,4,51,sensor_id4_q4
4,5,38,sensor_id3_q2
5,6,72,sensor_id6_q3
6,7,62,sensor_id5_q4
7,8,83,sensor_id7_q3
8,9,70,sensor_id6_q1
9,10,73,sensor_id6_q4
